In [1]:
import tensorflow  as tf

In [2]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

In [7]:
import os

DATA_DIR = r"C:\Users\Kalki Prathisha\Downloads\SkillDevelopment\AIML_Projects\Brain_Tumor_MRI_Image_Classification\Tumour"   

TRAIN_DIR = DATA_DIR + "/train"
VALID_DIR = DATA_DIR + "/valid"    
TEST_DIR  = DATA_DIR + "/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size= IMG_SIZE,
    batch_size= BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = False
)
test_ds  = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size= IMG_SIZE,
    batch_size= BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = False
)

class_names = train_ds.class_names
print("Class Names:", class_names)

Found 1695 files belonging to 4 classes.
Found 502 files belonging to 4 classes.
Found 246 files belonging to 4 classes.
Class Names: ['glioma', 'meningioma', 'no_tumor', 'pituitary']


In [5]:
normalizer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalizer(x),y)).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x,y:(normalizer(x),y)).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(lambda x,y:(normalizer(x),y)).prefetch(tf.data.AUTOTUNE)

In [6]:
for images,labels in train_ds.take(1):
    print("Batch image shape",images.shape)
    print("Batch label shape",labels.shape)
    print("First 5 labels",labels[:5].numpy())

Batch image shape (32, 224, 224, 1)
Batch label shape (32,)
First 5 labels [3 0 1 0 0]


### Build Baseline CNN

In [10]:
model = keras.Sequential([
    layers.Input(shape = (224,224,1)),
    layers.Conv2D(32,3,padding = "same",activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64,3,padding = "same",activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128,3,padding = "same",activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64,activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(4,activation = "softmax")

])

In [11]:
model.compile(
    optimizer = "adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = ["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,188 (395.27 KB)

 Trainable params: 101,188 (395.27 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 33s 548ms/step - accuracy: 0.4785 - loss: 1.2596 - val_accuracy: 0.5518 - val_loss: 1.0136
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 433ms/step - accuracy: 0.5587 - loss: 1.0134 - val_accuracy: 0.6653 - val_loss: 0.9513
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 431ms/step - accuracy: 0.6035 - loss: 0.9426 - val_accuracy: 0.6155 - val_loss: 0.9059
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 420ms/step - accuracy: 0.6460 - loss: 0.8901 - val_accuracy: 0.6713 - val_loss: 0.8525
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 422ms/step - accuracy: 0.6584 - loss: 0.8467 - val_accuracy: 0.7032 - val_loss: 0.7772
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 424ms/step - accuracy: 0.6749 - loss: 0.8137 - val_accuracy: 0.7211 - val_loss: 0.7446
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 42s 436ms/step - accuracy: 0.6926 - loss: 0.7858 - val_accuracy: 0.7191 - val_loss: 0.7571
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 23s 439ms/step - accuracy: 0.7015 - loss: 0.7600 - val_accu

In [13]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - accuracy: 0.7439 - loss: 0.7050
Test Accuracy: 0.7439


In [15]:
for images, labels in test_ds.take(1):
    preds = model.predict(images)              # shape: (batch, 4)
    pred_ids = np.argmax(preds, axis=1)        # predicted class index
    confidences = np.max(preds, axis=1)        # probability of predicted class

    print("\nSample Predictions:")
    for i in range(min(5, len(pred_ids))):
        true_label = class_names[labels[i].numpy()]
        pred_label = class_names[pred_ids[i]]
        conf = confidences[i]
        print(f"True: {true_label} | Pred: {pred_label} | Confidence: {conf:.3f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step

Sample Predictions:
True: glioma | Pred: meningioma | Confidence: 0.471
True: glioma | Pred: glioma | Confidence: 0.771
True: glioma | Pred: meningioma | Confidence: 0.378
True: glioma | Pred: meningioma | Confidence: 0.497
True: glioma | Pred: glioma | Confidence: 0.501
